In [1]:
%load_ext autoreload
%autoreload 2

import sys 
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd
from src import data, quality

pb = pd.read_parquet(data.INTERIM / "phrasebank.parquet")
tw = pd.read_parquet(data.INTERIM / "twitter.parquet")
print(pb.shape, tw.shape)   # Expected: (14780, 5) and (11931, 5)

train = tw[tw["source_file"] == "sent_train.csv"]
valid = tw[tw["source_file"] == "sent_valid.csv"]
print(len(train), len(valid))

(14780, 5) (11931, 5)
9543 2388


In [2]:

toy = pd.DataFrame({
    "text": ["Shares rise https://t.co/a", "shares  rise https://t.co/b",
             "Profit falls", "Profit falls", "New CEO named"],
    "label": [1, 1, 0, 2, 2],
})
d = quality.exact_duplicates(toy)
assert len(d) == 2
assert d.loc["profit falls", "n_labels"] == 2      # same text, two labels: label noise
assert d.loc["shares rise", "n_labels"] == 1       # repeated, consistent

ov = quality.cross_overlap(toy.iloc[:3], toy.iloc[3:])
assert list(ov["text"]) == ["Profit falls"]
print(d, "\nexact duplicates OK")

              n  n_labels
key                      
profit falls  2         2
shares rise   2         1 
exact duplicates OK


In [3]:
keys = quality.normalize_text(toy["text"])
print(keys.tolist())
# ['shares rise', 'shares rise', 'profit falls', 'profit falls', 'new ceo named']

rows = len(toy)
print("rows:", rows)                         # 5

unique_keys = keys.nunique()
print("unique_keys:", unique_keys)           # 3

repeated_keys = len(d)
print("repeated_keys:", repeated_keys)       # 2

rows_in_repeats = d["n"].sum()
print("rows_in_repeats:", rows_in_repeats)   # 4

conflict_keys = (d["n_labels"] > 1).sum()
print("conflict_keys:", conflict_keys)       # 1

['shares rise', 'shares rise', 'profit falls', 'profit falls', 'new ceo named']
rows: 5
unique_keys: 3
repeated_keys: 2
rows_in_repeats: 4
conflict_keys: 1


In [4]:
conflict_pct = conflict_keys / repeated_keys if repeated_keys > 0 else 0.0
print(f"conflict_pct: {conflict_pct:.1%}")

conflict_pct: 50.0%


In [5]:
clean_size = unique_keys - conflict_keys
print(f"clean_size: {clean_size}")

clean_size: 2


In [6]:
toy = pd.DataFrame({
    "text":      ["s1", "s2", "s1", "s2", "s3", "s1", "s2", "s3", "s4"],
    "label":     ["pos", "neg", "pos", "neg", "neu", "pos", "neu", "neu", "pos"],
    "agreement": [100, 100, 75, 75, 75, 50, 50, 50, 50],
})
n = quality.check_nesting(toy)
assert list(n["missing_in_looser"]) == [0, 0]
assert list(n["label_changes"]) == [0, 1]
print(n, "\ncheck_nesting OK")

   stricter  looser  n_stricter  missing_in_looser  label_changes
0       100      75           2                  0              0
1        75      50           3                  0              1 
check_nesting OK


In [8]:
def dup_summary(df: pd.DataFrame, label_col: str = "label") -> dict:
    """Duplicate counts for one dataset, or one PhraseBanks level."""
    d = quality.exact_duplicates(df, label_col=label_col)
    keys = quality.normalize_text(df["text"])

    unique_keys = keys.nunique()
    repeated_keys = len(d)
    conflict_keys = int((d["n_labels"] > 1).sum())

    return {
        "rows": len(df),
        "unique_keys": unique_keys,
        "repeated_keys": repeated_keys,
        "rows_in_repeats": int(d["n"].sum()),
        "conflict_keys": conflict_keys,
        "conflict_pct": conflict_keys / repeated_keys if repeated_keys > 0 else 0.0,
        "clean_size": unique_keys - conflict_keys,
    }

s = dup_summary(toy)
print(s)
# assert s["unique_keys"] == 3 and s["clean_size"] == 2 and s["conflict_pct"] == 0.5
print("dup_summary OK")

{'rows': 9, 'unique_keys': 4, 'repeated_keys': 3, 'rows_in_repeats': 8, 'conflict_keys': 1, 'conflict_pct': 0.3333333333333333, 'clean_size': 3}
dup_summary OK


In [9]:
results = []

# Twitter: the whole dataset, because cleaning happens before the split
results.append({"dataset": "twitter", **dup_summary(tw)})

# PhraseBank: one level at a time (never mix levels)
for level, g in pb.groupby("agreement"):
    results.append({"dataset": f"phrasebank_{level}", **dup_summary(g)})

summary = pd.DataFrame(results).set_index("dataset")
summary

,rows,unique_keys,repeated_keys,rows_in_repeats,conflict_keys,conflict_pct,clean_size
dataset,,,,,,,
twitter,11931,11779,98,250,3,0.030612,11776
phrasebank_50,4846,4838,8,16,2,0.250000,4836
phrasebank_66,4217,4211,6,12,0,0.000000,4211
phrasebank_75,3453,3448,5,10,0,0.000000,3448
phrasebank_100,2264,2259,5,10,0,0.000000,2259


In [10]:
leak = quality.cross_overlap(train, valid)
leak_pct = len(leak) / len(valid)
print(f"{len(leak)} of {len(valid)} valid rows ({leak_pct:.1%}) also appear in train")

39 of 2388 valid rows (1.6%) also appear in train


In [11]:
df = pb[pb["agreement"] == 50]
keyed = df.assign(key=quality.normalize_text(df["text"]))
d = quality.exact_duplicates(df)
conflict_keys = d[d["n_labels"] > 1].index      # the keys with conflict

conflicts = keyed[keyed["key"].isin(conflict_keys)].sort_values("key")
conflicts[["text", "label"]].head(40)

,text,label
10012,TELECOMWORLDWIRE-7 April 2006-TJ Group Plc sel...,neutral
10013,TELECOMWORLDWIRE-7 April 2006-TJ Group Plc sel...,positive
10722,The Group 's business is balanced by its broad...,positive
10723,The Group 's business is balanced by its broad...,neutral
